# Install

In [ ]:
!pip install --upgrade wandb

In [ ]:
import os

if not os.path.exists('Sketch-to-Image-by-Pix2Pix'):
    !git clone -b dev https://github.com/lqb464/Sketch-to-Image-by-Pix2Pix



from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

os.chdir('Sketch-to-Image-by-Pix2Pix/')
!pip install -q -r requirements.txt

# Training


In [ ]:
import yaml, random, numpy as np, torch

CONFIG_PATH = "./configs/model4_vgg19_pretrained.yaml"
# Giai doan 2: CONFIG_PATH = "./configs/model4_vgg_finetune.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SEED = cfg.get("seed", 42)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f"Config         : {CONFIG_PATH}")
print(f"Experiment name: {cfg['name']}")
print(f"netG           : {cfg['netG']}")
print(f"freeze_encoder : {cfg.get('freeze_encoder', True)}")
print(f"Epochs         : {cfg['n_epochs']} + {cfg['n_epochs_decay']} = {cfg['n_epochs'] + cfg['n_epochs_decay']}")
print(f"lr             : {cfg['lr']}")
print(f"Seed           : {SEED}")

In [ ]:
# Cac flag boolean (store_true) xu ly rieng
freeze_flag     = "--freeze_encoder" if cfg.get("freeze_encoder", True)  else ""
no_dropout_flag = "--no_dropout"      if cfg.get("no_dropout", False)      else ""
continue_flag   = "--continue_train"  if cfg.get("continue_train", False)  else ""
flags = " ".join(f for f in [freeze_flag, no_dropout_flag, continue_flag, "--no_html"] if f)

train_cmd = (
    f'python train.py'
    f' --dataroot         /kaggle/input/datasets/lqb464/sketch2image-dataset/pipeline_output/output/airplane'
    f' --name             {cfg["name"]}'
    f' --model            {cfg["model"]}'
    f' --direction        {cfg["direction"]}'
    f' --netG             {cfg["netG"]}'
    f' --netD             {cfg["netD"]}'
    f' --norm             {cfg["norm"]}'
    f' --gan_mode         {cfg["gan_mode"]}'
    f' --lambda_L1        {cfg["lambda_L1"]}'
    f' --batch_size       {cfg["batch_size"]}'
    f' --lr               {cfg["lr"]}'
    f' --lr_policy        {cfg.get("lr_policy", "linear")}'
    f' --beta1            {cfg.get("beta1", 0.5)}'
    f' --n_epochs         100'
    f' --n_epochs_decay   100'
    f' --num_threads      {cfg["num_threads"]}'
    f' --load_size        {cfg["load_size"]}'
    f' --crop_size        {cfg["crop_size"]}'
    f' --print_freq       {cfg["print_freq"]}'
    f' --display_freq     {cfg["display_freq"]}'
    f' --save_latest_freq {cfg["save_latest_freq"]}'
    f' --save_epoch_freq  {cfg["save_epoch_freq"]}'
    f' --dataset_mode     {cfg.get("dataset_mode", "aligned")}'
    f' --use_wandb'
    f' {flags}'
)

In [ ]:
with open("util/visualizer.py") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if 'wandb.init(' in line and 'self.wandb_run' in line:
        indent = ' ' * 16
        new_lines.append(indent + 'wandb.login(key=os.environ.get("WANDB_API_KEY"), relogin=True)\n')
    new_lines.append(line)

with open("util/visualizer.py", "w") as f:
    f.writelines(new_lines)
print("Done")

In [ ]:
!{train_cmd}

In [ ]:
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("WANDB_API_KEY")
print(len(key))
print(key)

# Testing

Chay inference tren test set. `num_test` lay tu config (mac dinh 50).

In [ ]:
num_test    = cfg.get("num_test", 100)
freeze_flag = "--freeze_encoder" if cfg.get("freeze_encoder", True) else ""

test_cmd = (
    f'python test.py'
    f' --dataroot     /kaggle/input/datasets/lqb464/sketch2image-dataset/pipeline_output/output/airplane'
    f' --name         {cfg["name"]}'
    f' --model        {cfg["model"]}'
    f' --direction    {cfg["direction"]}'
    f' --netG         {cfg["netG"]}'
    f' --norm         {cfg["norm"]}'
    f' --load_size    {cfg["load_size"]}'
    f' --crop_size    {cfg["crop_size"]}'
    f' --num_test     {num_test}'
    f' --dataset_mode {cfg.get("dataset_mode", "aligned")}'
    f' {freeze_flag}'
    f' --use_wandb'
)

In [ ]:
!{test_cmd}

# Visualize

In [ ]:
import matplotlib.pyplot as plt
import os

base = f"./results/{cfg['name']}/test_latest/images/"

fake_files = sorted([f for f in os.listdir(base) if f.endswith("_fake_B.png")])[:20]

fig, axes = plt.subplots(len(fake_files), 3, figsize=(12, 4 * len(fake_files)))
if len(fake_files) == 1:
    axes = [axes]

for ax_row, fname in zip(axes, fake_files):
    stem   = fname.replace("_fake_B.png", "")
    real_a = base + stem + "_real_A.png"
    fake_b = base + fname
    real_b = base + stem + "_real_B.png"

    for ax, path, title in zip(
        ax_row,
        [real_a, fake_b, real_b],
        ["Input (Sketch A)", "Generated (Fake B)", "Ground Truth (Real B)"]
    ):
        ax.imshow(plt.imread(path))
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.savefig(f"./results/{cfg['name']}_preview_20.png", dpi=100, bbox_inches="tight")
plt.show()